In [46]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Read data from fz1 - FZ1.1

In [47]:
# First read lines 8 and 9 to check which one contains the header
header_check = pd.read_excel('../../Data prep/Bestand/01 Kraftfahrzeuge und Kraftfahrzeuganhänger nach Zulassungsbezirken (FZ 1)/fz1_2025.xlsx', 
                            sheet_name='FZ1.1', 
                            skiprows=7, 
                            nrows=2)

In [48]:
# Check if line 8 (first row) is empty
if header_check.iloc[1].isna().all():
    header_row = 8  # Use line 9 as header
else:
    header_row = 7  # Use line 8 as header

# print(f"Using row {header_row + 1} as header")

In [49]:
# Now read the full file with the correct header row
df = pd.read_excel('../../Data prep/Bestand/01 Kraftfahrzeuge und Kraftfahrzeuganhänger nach Zulassungsbezirken (FZ 1)/fz1_2025.xlsx', 
                   sheet_name='FZ1.1', 
                   header=header_row)


In [50]:
df = df.iloc[:, 1:]

In [51]:
# Check if there are any unnamed columns
if df.columns.str.contains('^Unnamed').any():
    first_row = df.iloc[0]

    # Replace only the unnamed headers
    new_columns = [
        first_row[i] if col.startswith('Unnamed') else col
        for i, col in enumerate(df.columns)
    ]

    df.columns = new_columns
    df = df[1:].reset_index(drop=True)  # Drop the first row (used as partial header)

C:\Users\janta\AppData\Local\Temp\ipykernel_1688\1403245063.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_row[i] if col.startswith('Unnamed') else col


In [52]:
# Forward fill multiple columns that might have merged cells
columns_to_fill = ['Land', 'Regierungsbezirk']  # Add any other columns that have merged cells
for col in columns_to_fill:
    df[col] = df[col].fillna(method='ffill')

# Display the results
# print("First 10 rows of the DataFrame:")
# print(df[columns_to_fill + ['Unnamed: 3']].head(10))

C:\Users\janta\AppData\Local\Temp\ipykernel_1688\1868915817.py:4: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='ffill')


In [53]:
# Remove rows where column 1 contains "zusammen"
df = df[~df.iloc[:, 1].astype(str).str.contains('zusammen', case=False, na=False)].reset_index(drop=True)


In [54]:
df.columns = df.columns.str.replace('\n', ' ', regex=False)
df.columns = df.columns.str.strip()

In [55]:
df.head()

,Land,Regierungsbezirk,Statistische Kennziffer und Zulassungsbezirk,Krafträder,davon zweirädrige Kfz,davon dreirädrige Kfz,davon leichte vierrädrige Kfz,darunter Halterinnen,Personenkraftwagen,Hubraum bis 1.399 cm³,...,unbe- kannt,Zugmaschinen,davon Sattelzug- maschinen,davon land-/forst- wirtschaft- liche Zug- maschinen,davon sonstige Zug- maschinen,Sonstige Kfz,Nutzfahr- zeuge insgesamt,Kraftfahrzeuge,Kfz-Dichte je 1.000 Einwohner,Kraftfahrzeug- anhänger
0,BADEN-WUERTTEMBERG,RB STUTTGART,"08111 STUTTGART,STADT",28548,27812,537,199,3789,301985,93124,...,9,3131,798,1906,427,2590.0,24545.0,355078,561,19716.0
1,BADEN-WUERTTEMBERG,RB STUTTGART,08115 BOEBLINGEN,26719,26075,393,251,3591,265649,84848,...,7,7938,508,5306,2124,946.0,22537.0,314905,785,31448.0
2,BADEN-WUERTTEMBERG,RB STUTTGART,08116 ESSLINGEN,40747,39889,526,332,6221,345526,120137,...,10,11518,1077,6969,3472,1531.0,34321.0,420594,775,45978.0
3,BADEN-WUERTTEMBERG,RB STUTTGART,08117 GOEPPINGEN,19795,19324,229,242,3074,171708,60995,...,4,8579,477,5860,2242,723.0,20923.0,212426,806,27297.0
4,BADEN-WUERTTEMBERG,RB STUTTGART,08118 LUDWIGSBURG,38394,37559,528,307,5456,342151,118705,...,15,11949,1314,7903,2732,1460.0,35024.0,415569,751,43395.0


In [56]:
# Divide columns if needed
df[['Statistische Kennziffer', 'Zulassungsbezirk']] = df['Statistische Kennziffer und Zulassungsbezirk'].str.extract(r'(\d{5})\s+(.+)')

In [57]:
# Divide columns if needed
df['Regierungs_Bezirk'] = df['Regierungsbezirk'].str.replace(r'^[A-ZÄÖÜa-zäöü]{2}\s+', '', regex=True)


In [58]:
df.head()

,Land,Regierungsbezirk,Statistische Kennziffer und Zulassungsbezirk,Krafträder,davon zweirädrige Kfz,davon dreirädrige Kfz,davon leichte vierrädrige Kfz,darunter Halterinnen,Personenkraftwagen,Hubraum bis 1.399 cm³,...,davon land-/forst- wirtschaft- liche Zug- maschinen,davon sonstige Zug- maschinen,Sonstige Kfz,Nutzfahr- zeuge insgesamt,Kraftfahrzeuge,Kfz-Dichte je 1.000 Einwohner,Kraftfahrzeug- anhänger,Statistische Kennziffer,Zulassungsbezirk,Regierungs_Bezirk
0,BADEN-WUERTTEMBERG,RB STUTTGART,"08111 STUTTGART,STADT",28548,27812,537,199,3789,301985,93124,...,1906,427,2590.0,24545.0,355078,561,19716.0,08111,"STUTTGART,STADT",STUTTGART
1,BADEN-WUERTTEMBERG,RB STUTTGART,08115 BOEBLINGEN,26719,26075,393,251,3591,265649,84848,...,5306,2124,946.0,22537.0,314905,785,31448.0,08115,BOEBLINGEN,STUTTGART
2,BADEN-WUERTTEMBERG,RB STUTTGART,08116 ESSLINGEN,40747,39889,526,332,6221,345526,120137,...,6969,3472,1531.0,34321.0,420594,775,45978.0,08116,ESSLINGEN,STUTTGART
3,BADEN-WUERTTEMBERG,RB STUTTGART,08117 GOEPPINGEN,19795,19324,229,242,3074,171708,60995,...,5860,2242,723.0,20923.0,212426,806,27297.0,08117,GOEPPINGEN,STUTTGART
4,BADEN-WUERTTEMBERG,RB STUTTGART,08118 LUDWIGSBURG,38394,37559,528,307,5456,342151,118705,...,7903,2732,1460.0,35024.0,415569,751,43395.0,08118,LUDWIGSBURG,STUTTGART


## Read data from fz1 - FZ1.2

In [59]:
# First read lines 8 and 9 to check which one contains the header
header_check_2 = pd.read_excel('../../Data prep/Bestand/01 Kraftfahrzeuge und Kraftfahrzeuganhänger nach Zulassungsbezirken (FZ 1)/fz1_2025.xlsx', 
                            sheet_name='FZ1.2', 
                            skiprows=7, 
                            nrows=2)

In [60]:
# Check if line 8 (first row) is empty
if header_check_2.iloc[1].isna().all():
    header_row = 8  # Use line 9 as header
else:
    header_row = 7  # Use line 8 as header

# print(f"Using row {header_row + 1} as header")

In [61]:
# Now read the full file with the correct header row
df_2 = pd.read_excel('../../Data prep/Bestand/01 Kraftfahrzeuge und Kraftfahrzeuganhänger nach Zulassungsbezirken (FZ 1)/fz1_2025.xlsx', 
                   sheet_name='FZ1.2', 
                   header=header_row)


In [62]:
df_2

,Unnamed: 0,Land\n\n,Regierungsbezirk,Statistische Kennziffer und Zulassungsbezirk\n,Insgesamt,Nach Kraftstoffarten,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33
0,NaN,NaN,NaN,NaN,NaN,Benzin,Diesel,Gas\n(einschl.\nbivalent),Hybrid \ninsgesamt,darunter\nPlug-in-Hybrid,...,Euro 2,Euro 3,Euro 4,Euro 5,Euro 6,darunter Euro 6d-temp,darunter Euro 6d,darunter Euro 6e,sonstige,schadstoffred. mit Dieselantrieb insgesamt
1,NaN,BADEN-WUERTTEMBERG,RB STUTTGART,"08111 STUTTGART,STADT",301985.0,180256,61062,1792,40884,17351,...,968,2052,2455,10121,44398,8351,15308,2934,168,60272
2,NaN,NaN,NaN,08115 BOEBLINGEN,265649.0,152148,62141,1408,33957,12354,...,1281,3352,6021,15756,34723,6863,8704,1212,187,61474
3,NaN,NaN,NaN,08116 ESSLINGEN,345526.0,213666,83821,1883,31239,9547,...,1908,4808,7384,20781,47437,9395,13240,2123,281,82896
4,NaN,NaN,NaN,08117 GOEPPINGEN,171708.0,105558,46770,1401,12315,3490,...,1061,2766,4792,12942,24479,5007,5960,732,117,46298
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
444,NaN,THUERINGEN INSGESAMT,NaN,NaN,1192603.0,752426,337400,6802,73742,15507,...,6702,17007,27662,96426,186458,40353,39132,3732,658,335728
445,NaN,SONSTIGE,NaN,NaN,47687.0,22592,19246,374,3548,1146,...,494,1296,1303,5811,10157,2306,1900,120,64,19168
446,NaN,DEUTSCHLAND INSGESAMT,NaN,NaN,49339166.0,29918520,13829261,373045,3556956,967423,...,313793,817526,1468590,4017497,7009951,1413606,1595992,201153,44399,13716887
447,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [63]:
df_2 = df_2.iloc[:, 1:]

In [64]:
df_2

,Land\n\n,Regierungsbezirk,Statistische Kennziffer und Zulassungsbezirk\n,Insgesamt,Nach Kraftstoffarten,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,...,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33
0,NaN,NaN,NaN,NaN,Benzin,Diesel,Gas\n(einschl.\nbivalent),Hybrid \ninsgesamt,darunter\nPlug-in-Hybrid,Elektro (BEV),...,Euro 2,Euro 3,Euro 4,Euro 5,Euro 6,darunter Euro 6d-temp,darunter Euro 6d,darunter Euro 6e,sonstige,schadstoffred. mit Dieselantrieb insgesamt
1,BADEN-WUERTTEMBERG,RB STUTTGART,"08111 STUTTGART,STADT",301985.0,180256,61062,1792,40884,17351,17944,...,968,2052,2455,10121,44398,8351,15308,2934,168,60272
2,NaN,NaN,08115 BOEBLINGEN,265649.0,152148,62141,1408,33957,12354,15953,...,1281,3352,6021,15756,34723,6863,8704,1212,187,61474
3,NaN,NaN,08116 ESSLINGEN,345526.0,213666,83821,1883,31239,9547,14856,...,1908,4808,7384,20781,47437,9395,13240,2123,281,82896
4,NaN,NaN,08117 GOEPPINGEN,171708.0,105558,46770,1401,12315,3490,5637,...,1061,2766,4792,12942,24479,5007,5960,732,117,46298
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
444,THUERINGEN INSGESAMT,NaN,NaN,1192603.0,752426,337400,6802,73742,15507,21921,...,6702,17007,27662,96426,186458,40353,39132,3732,658,335728
445,SONSTIGE,NaN,NaN,47687.0,22592,19246,374,3548,1146,1918,...,494,1296,1303,5811,10157,2306,1900,120,64,19168
446,DEUTSCHLAND INSGESAMT,NaN,NaN,49339166.0,29918520,13829261,373045,3556956,967423,1651643,...,313793,817526,1468590,4017497,7009951,1413606,1595992,201153,44399,13716887
447,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [65]:
# Check if there are any unnamed columns in the header
if df_2.columns.str.contains('^Unnamed').any():
    first_row = df_2.iloc[0]

    # Replace only the unnamed headers
    new_columns = [
        first_row[i] if col.startswith('Unnamed') else col
        for i, col in enumerate(df_2.columns)
    ]

    df_2.columns = new_columns
    df_2 = df_2[1:].reset_index(drop=True)  # Drop the first row (used as partial header)

C:\Users\janta\AppData\Local\Temp\ipykernel_1688\1054534435.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_row[i] if col.startswith('Unnamed') else col


In [66]:
df_2

,Land\n\n,Regierungsbezirk,Statistische Kennziffer und Zulassungsbezirk\n,Insgesamt,Nach Kraftstoffarten,Diesel,Gas\n(einschl.\nbivalent),Hybrid \ninsgesamt,darunter\nPlug-in-Hybrid,Elektro (BEV),...,Euro 2,Euro 3,Euro 4,Euro 5,Euro 6,darunter Euro 6d-temp,darunter Euro 6d,darunter Euro 6e,sonstige,schadstoffred. mit Dieselantrieb insgesamt
0,BADEN-WUERTTEMBERG,RB STUTTGART,"08111 STUTTGART,STADT",301985.0,180256,61062,1792,40884,17351,17944,...,968,2052,2455,10121,44398,8351,15308,2934,168,60272
1,NaN,NaN,08115 BOEBLINGEN,265649.0,152148,62141,1408,33957,12354,15953,...,1281,3352,6021,15756,34723,6863,8704,1212,187,61474
2,NaN,NaN,08116 ESSLINGEN,345526.0,213666,83821,1883,31239,9547,14856,...,1908,4808,7384,20781,47437,9395,13240,2123,281,82896
3,NaN,NaN,08117 GOEPPINGEN,171708.0,105558,46770,1401,12315,3490,5637,...,1061,2766,4792,12942,24479,5007,5960,732,117,46298
4,NaN,NaN,08118 LUDWIGSBURG,342151.0,222706,76142,1939,28184,8811,13127,...,1743,4449,6856,19184,42890,8770,10448,1295,140,75412
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
443,THUERINGEN INSGESAMT,NaN,NaN,1192603.0,752426,337400,6802,73742,15507,21921,...,6702,17007,27662,96426,186458,40353,39132,3732,658,335728
444,SONSTIGE,NaN,NaN,47687.0,22592,19246,374,3548,1146,1918,...,494,1296,1303,5811,10157,2306,1900,120,64,19168
445,DEUTSCHLAND INSGESAMT,NaN,NaN,49339166.0,29918520,13829261,373045,3556956,967423,1651643,...,313793,817526,1468590,4017497,7009951,1413606,1595992,201153,44399,13716887
446,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [67]:
# Adjust header format
df_2.columns = df_2.columns.str.replace('\n', ' ', regex=False)
df_2.columns = df_2.columns.str.replace('\t', ' ', regex=False)
df_2.columns = df_2.columns.str.strip()
# Check the columns
# print(df.columns.tolist())

In [68]:
df_2

,Land,Regierungsbezirk,Statistische Kennziffer und Zulassungsbezirk,Insgesamt,Nach Kraftstoffarten,Diesel,Gas (einschl. bivalent),Hybrid insgesamt,darunter Plug-in-Hybrid,Elektro (BEV),...,Euro 2,Euro 3,Euro 4,Euro 5,Euro 6,darunter Euro 6d-temp,darunter Euro 6d,darunter Euro 6e,sonstige,schadstoffred. mit Dieselantrieb insgesamt
0,BADEN-WUERTTEMBERG,RB STUTTGART,"08111 STUTTGART,STADT",301985.0,180256,61062,1792,40884,17351,17944,...,968,2052,2455,10121,44398,8351,15308,2934,168,60272
1,NaN,NaN,08115 BOEBLINGEN,265649.0,152148,62141,1408,33957,12354,15953,...,1281,3352,6021,15756,34723,6863,8704,1212,187,61474
2,NaN,NaN,08116 ESSLINGEN,345526.0,213666,83821,1883,31239,9547,14856,...,1908,4808,7384,20781,47437,9395,13240,2123,281,82896
3,NaN,NaN,08117 GOEPPINGEN,171708.0,105558,46770,1401,12315,3490,5637,...,1061,2766,4792,12942,24479,5007,5960,732,117,46298
4,NaN,NaN,08118 LUDWIGSBURG,342151.0,222706,76142,1939,28184,8811,13127,...,1743,4449,6856,19184,42890,8770,10448,1295,140,75412
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
443,THUERINGEN INSGESAMT,NaN,NaN,1192603.0,752426,337400,6802,73742,15507,21921,...,6702,17007,27662,96426,186458,40353,39132,3732,658,335728
444,SONSTIGE,NaN,NaN,47687.0,22592,19246,374,3548,1146,1918,...,494,1296,1303,5811,10157,2306,1900,120,64,19168
445,DEUTSCHLAND INSGESAMT,NaN,NaN,49339166.0,29918520,13829261,373045,3556956,967423,1651643,...,313793,817526,1468590,4017497,7009951,1413606,1595992,201153,44399,13716887
446,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [69]:
# Forward fill multiple columns that might have merged cells
columns_to_fill = ['Land', 'Regierungsbezirk']  # Add any other columns that have merged cells
for col in columns_to_fill:
    df_2[col] = df_2[col].fillna(method='ffill')

# Display the results
# print("First 10 rows of the DataFrame:")
# print(df[columns_to_fill + ['Unnamed: 3']].head(10))

C:\Users\janta\AppData\Local\Temp\ipykernel_1688\2659603149.py:4: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_2[col] = df_2[col].fillna(method='ffill')


In [70]:
# Remove rows where column 1 contains "zusammen"
df_2 = df_2[~df_2.iloc[:, 1].astype(str).str.contains('zusammen', case=False, na=False)].reset_index(drop=True)


In [ ]:
df_2.head(20)

In [ ]:
# Define columns for extraction
source_column = 'Statistische Kennziffer und Zulassungsbezirk'
target_columns = ['Statistische Kennziffer', 'Zulassungsbezirk']
first_part_length = 5  # Number of characters for the first part

# Create the pattern dynamically
pattern = f'(\\d{{{first_part_length}}})\\s+(.+)'

# Extract and split the data
df_2[target_columns] = df_2[source_column].str.extract(pattern)

In [ ]:
# Define columns for extraction
source_column = 'Regierungsbezirk'
target_column = 'Regierungs_Bezirk'  # Changed from list to string
first_part_length = 2  # Number of characters for the first part

# Create the pattern to remove the first part
pattern = f'^[A-ZÄÖÜa-zäöü]{{{first_part_length}}}\\s+'  # Match start of string, letters, and whitespace

# Remove the first part and keep the rest
df_2[target_column] = df_2[source_column].str.replace(pattern, '', regex=True) 

In [71]:
def extract_columns(df, source_column, target_columns, first_part_length, is_numeric=True):
    """
    Extract and split data from a source column into target columns.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The dataframe to modify
    source_column : str
        The source column name containing the data to split
    target_columns : str or list
        The target column name(s) for the extracted data
    first_part_length : int
        Number of characters for the first part
    is_numeric : bool
        If True, first part is numeric (\\d), if False, first part is letters ([A-ZÄÖÜa-zäöü])
    
    Returns:
    --------
    pandas.DataFrame
        The modified dataframe
    """
    # Convert single target column to list for consistent handling
    if isinstance(target_columns, str):
        target_columns = [target_columns]
    
    # Create pattern based on whether we're extracting one or two columns
    if len(target_columns) == 1:
        # For single column extraction, remove the first part
        pattern_type = '\\d' if is_numeric else '[A-ZÄÖÜa-zäöü]'
        pattern = f'^{pattern_type}{{{first_part_length}}}\\s+'
        df[target_columns[0]] = df[source_column].str.replace(pattern, '', regex=True)
    else:
        # For two column extraction, capture both parts
        pattern_type = '\\d' if is_numeric else '[A-ZÄÖÜa-zäöü]'
        pattern = f'({pattern_type}{{{first_part_length}}})\\s+(.+)'
        df[target_columns] = df[source_column].str.extract(pattern)
    
    return df

# Example usage:
# For numeric first part (5 digits) and two columns:
# df_2 = extract_columns(df_2, 'Statistische Kennziffer und Zulassungsbezirk', 
#                       ['Statistische Kennziffer', 'Zulassungsbezirk'], 5, is_numeric=True)

# For letter first part (2 letters) and single column:
# df_2 = extract_columns(df_2, 'Regierungsbezirk', 'Regierungs_Bezirk', 2, is_numeric=False) 

In [72]:
# Example usage:
# For numeric first part (5 digits) and two columns:
df_2 = extract_columns(df_2, 'Statistische Kennziffer und Zulassungsbezirk', 
                       ['Statistische Kennziffer', 'Zulassungsbezirk'], 5, is_numeric=True)

In [73]:
df_2.head()

,Land,Regierungsbezirk,Statistische Kennziffer und Zulassungsbezirk,Insgesamt,Nach Kraftstoffarten,Diesel,Gas (einschl. bivalent),Hybrid insgesamt,darunter Plug-in-Hybrid,Elektro (BEV),...,Euro 4,Euro 5,Euro 6,darunter Euro 6d-temp,darunter Euro 6d,darunter Euro 6e,sonstige,schadstoffred. mit Dieselantrieb insgesamt,Statistische Kennziffer,Zulassungsbezirk
0,BADEN-WUERTTEMBERG,RB STUTTGART,"08111 STUTTGART,STADT",301985.0,180256,61062,1792,40884,17351,17944,...,2455,10121,44398,8351,15308,2934,168,60272,08111,"STUTTGART,STADT"
1,BADEN-WUERTTEMBERG,RB STUTTGART,08115 BOEBLINGEN,265649.0,152148,62141,1408,33957,12354,15953,...,6021,15756,34723,6863,8704,1212,187,61474,08115,BOEBLINGEN
2,BADEN-WUERTTEMBERG,RB STUTTGART,08116 ESSLINGEN,345526.0,213666,83821,1883,31239,9547,14856,...,7384,20781,47437,9395,13240,2123,281,82896,08116,ESSLINGEN
3,BADEN-WUERTTEMBERG,RB STUTTGART,08117 GOEPPINGEN,171708.0,105558,46770,1401,12315,3490,5637,...,4792,12942,24479,5007,5960,732,117,46298,08117,GOEPPINGEN
4,BADEN-WUERTTEMBERG,RB STUTTGART,08118 LUDWIGSBURG,342151.0,222706,76142,1939,28184,8811,13127,...,6856,19184,42890,8770,10448,1295,140,75412,08118,LUDWIGSBURG
